# සිංහල / General AI Chatbot — Colab Runner (Stable / SFT)

විශ්වාසදායක, error අඩු version එක. ORPO (fragile) ඉවත් කලා — SFT පමණයි.

**මුලට:** `Runtime` → `Change runtime type` → `GPU` (T4) → Save. ඊට පස්සේ Run all.

In [6]:
# 1) Packages install (Unsloth = වේගවත් 4-bit training)
!pip install -q unsloth trl peft accelerate bitsandbytes datasets transformers gradio openai huggingface_hub
print('✅ packages installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
# 2) Repo clone + cd
!git clone https://github.com/hirushanethsara358-bot/qlora-sinhala-task.git
%cd qlora-sinhala-task
!ls

## 3) Proper SFT training (public datasets mix)

UltraChat (general chat) + SlimOrca (reasoning) mix කරලා SFT. විශ්වාසදායක සහ බලවත්.
Free T4 එකේ විනාඩි 20–60. (`--max_steps` වැඩි කරන්න නම් වඩා හොඳ).

In [ ]:
!python train_hf_dataset.py \
    --datasets "HuggingFaceH4/ultrachat_200k,Open-Orca/SlimOrca" \
    --sample 3000 --r 32 --max_steps 300

## 4) Trained model එක HF Hub එකට save කරන්න (IMPORTANT)

Colab session බිඳුණොත් `outputs/` නැතිවේ. HF account token (`hf_...`) දෙන්න.

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_xxxx'   # <-- ඔයාගේ Hugging Face token
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(folder_path='outputs',
                   repo_id='YOUR_USERNAME/sinhala-general-chatbot',  # <-- username
                   token=os.environ['HF_TOKEN'])
print('✅ pushed to Hugging Face Hub')

## 5) Chat (Gradio public link)

Train කලේ නම් `--lora outputs`; නැත්නම් base model.

In [ ]:
!python chat.py --lora outputs 2>/dev/null || python chat.py